# Notebook 14 — Feature Engineering v3 + Re-Tuning

**Goal:** Add 5 new features to the 38-feature set, run ablation to confirm contribution, then re-tune LGBM with a lower `reg_alpha` ceiling (0.5–10.0 vs NB12's 17.28) to allow compound-interaction features to contribute. Also run a Pre-Season Testing ablation.

**Gap to close:** Best LB = 0.92845. Leaderboard top = 0.95488. Need CV AUC ~0.930 (current best: 0.9032).

**New features (Tier 7):**

| Feature | Type | Rationale |
|---------|------|-----------|
| `TyreLife_cliff_proximity` | Non-fold-aware | Signed distance from compound cliff: negative = normal, positive = overdue. Dry only (wet → 0). |
| `position_trend_slope` | Non-fold-aware | OLS slope of Position over last 5 laps — direction of position change, complements `pos_change_rolling_std_3` |
| `TyreLife_x_cliff_proximity` | Non-fold-aware | Quadratic interaction: urgency escalates nonlinearly past cliff |
| `Driver_stint_pct` | Fold-aware | TyreLife / driver's median typical stint length for this compound — ratio form of TyreLife_vs_driver_typical |
| `Race_Compound_target_encoded` | Fold-aware | Mean pit rate per (Race, Compound) — circuit+compound strategy (~60 groups, less fragmentation than Race×Year's 104) |

**Inputs:** `data/processed/train_features_v2.parquet`, `data/processed/test_features_v2.parquet`, `data/processed/fold_assignments.parquet`  
**Outputs:** `data/processed/train_features_v3.parquet`, `data/processed/test_features_v3.parquet`, `data/processed/oof_predictions_nb14.parquet`, `data/processed/test_predictions_nb14.parquet`, `models/nb14_summary.pkl`

**Self-contained:** all helpers defined inline. Runs on Kaggle without modification.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import time
import warnings
import pickle
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print('WARNING: catboost not installed — CatBoost cells will be skipped')

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
print(f'LightGBM  : {lgb.__version__}')
print(f'CatBoost  : {"available" if CATBOOST_AVAILABLE else "NOT installed"}')
print('Imports OK')

c:\Repos\predict-f1-pit-stops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LightGBM  : 4.6.0
CatBoost  : available
Imports OK


In [2]:
# Robust project root detection — works from workspace root, notebooks/, or Kaggle
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root  = cwd
data_dir      = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

# Kaggle path fallback
if not data_dir.exists():
    data_dir      = Path('/kaggle/input/playground-series-s6e5')
    processed_dir = Path('/kaggle/working')

print(f'Project root : {project_root}')
print(f'Processed dir: {processed_dir}')

train = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test  = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds = pd.read_parquet(processed_dir / 'fold_assignments.parquet')

train = train.merge(folds, on='id', how='left')

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Fold distribution:\n{train["fold"].value_counts().sort_index().to_string()}')

Project root : c:\Repos\predict-f1-pit-stops
Processed dir: c:\Repos\predict-f1-pit-stops\data\processed
Train: (439140, 53)  |  Test: (188165, 51)
Fold distribution:
fold
0    88018
1    87444
2    88027
3    87854
4    87797


## Tier 7 Feature Engineering — Non-Fold-Aware Features

Three features computed directly (no target leakage risk):
- `TyreLife_cliff_proximity`: signed distance from compound-specific cliff. Cliff thresholds from EDA: SOFT=13, MED=49, HARD=61. For wet tyres, set to 0 (no cliff concept for INTERMEDIATE/WET).
- `position_trend_slope`: OLS slope of Position over last 5 laps. Positive = consistently losing positions = tyre degradation signal. Grouped by `(Race, Year, Driver)` (not Stint — same rationale as `pos_change_rolling_std_3`).
- `TyreLife_x_cliff_proximity`: interaction term capturing quadratic escalation past cliff.

In [3]:
# Compute Tier 7 non-fold-aware features on train and test
CLIFF = {'SOFT': 13, 'MEDIUM': 49, 'HARD': 61, 'INTERMEDIATE': 1e6, 'WET': 1e6}
DRIVER_KEYS = ['Race', 'Year', 'Driver']

def add_tier7_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add 3 new non-fold-aware Tier 7 features. Input must have all v2 columns."""
    df = df.copy().sort_values(['Race', 'Year', 'LapNumber']).reset_index(drop=True)

    # TyreLife_cliff_proximity: signed distance from compound cliff (dry only)
    cliff_vals = df['Compound'].map(CLIFF).values.astype(float)
    proximity_raw = df['TyreLife'].values / cliff_vals - 1.0
    # Wet tyres (is_wet_tyre=1) get 0 — no cliff concept applies
    df['TyreLife_cliff_proximity'] = np.where(
        df['is_wet_tyre'].values == 0, proximity_raw, 0.0
    )

    # position_trend_slope: OLS slope of Position over last 5 laps
    # x = [0,1,2,3,4] (lag4=oldest, lag0=current), closed-form slope formula:
    # slope = (sum_xy - n*mean_x*mean_y) / (sum_x2 - n*mean_x^2)
    # For x=[0..4]: sum_x=10, mean_x=2, sum_x2=30, denominator=10
    # → slope = (0*y4 + 1*y3 + 2*y2 + 3*y1 + 4*y0 - 10*mean_y) / 10
    # Groups by (Race,Year,Driver) only — intentionally spans stints
    grp = df.groupby(DRIVER_KEYS)['Position']
    pos_cur = df['Position'].values.astype(float)
    lag1 = grp.shift(1).fillna(df['Position']).values.astype(float)
    lag2 = grp.shift(2).fillna(df['Position']).values.astype(float)
    lag3 = grp.shift(3).fillna(df['Position']).values.astype(float)
    lag4 = grp.shift(4).fillna(df['Position']).values.astype(float)
    sum_xy = (0*lag4 + 1*lag3 + 2*lag2 + 3*lag1 + 4*pos_cur)
    mean_y = (lag4 + lag3 + lag2 + lag1 + pos_cur) / 5.0
    df['position_trend_slope'] = (sum_xy - 10.0 * mean_y) / 10.0

    # Interaction: quadratic escalation past cliff
    df['TyreLife_x_cliff_proximity'] = df['TyreLife'] * df['TyreLife_cliff_proximity']

    return df


t0 = time.time()
train = add_tier7_features(train)
test  = add_tier7_features(test)
print(f'Done in {time.time()-t0:.1f}s')

TIER7_NON_ENC = ['TyreLife_cliff_proximity', 'position_trend_slope', 'TyreLife_x_cliff_proximity']

# Signal check
print('\nNew feature correlations with PitNextLap:')
for f in TIER7_NON_ENC:
    corr = train[f].corr(train['PitNextLap'])
    print(f'  {f:<35} {corr:+.4f}')

# Null check
nulls = train[TIER7_NON_ENC].isnull().sum()
print(f'\nNull check: {"PASSED" if not nulls.any() else nulls[nulls>0]}')

# Cliff proximity stats
print('\nTyreLife_cliff_proximity by compound (dry only):')
dry = train[train['is_wet_tyre'] == 0]
print(dry.groupby('Compound')['TyreLife_cliff_proximity'].describe()[['mean','min','50%','max']].round(3))

Done in 0.5s

New feature correlations with PitNextLap:
  TyreLife_cliff_proximity            +0.1186
  position_trend_slope                +0.0040
  TyreLife_x_cliff_proximity          -0.1735

Null check: PASSED

TyreLife_cliff_proximity by compound (dry only):
           mean    min    50%    max
Compound                            
HARD     -0.707 -0.984 -0.721  0.262
MEDIUM   -0.759 -0.980 -0.776  0.551
SOFT     -0.140 -0.923 -0.231  4.077


## Fold-Aware Target Encodings v3

Extends `apply_target_encodings()` from NB12 with 2 new fold-aware features:
- `Driver_stint_pct`: TyreLife / driver's median typical stint length for this compound (ratio). Captures "at what fraction of their typical stint is this driver running?"
- `Race_Compound_target_encoded`: mean pit rate per (Race, Compound) — ~60 groups vs 26 Race-only or 104 Race×Year groups.

In [4]:
def apply_target_encodings_v3(train_df: pd.DataFrame, val_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute 5 fold-aware encodings from train_df; apply to val_df.
    Call INSIDE each CV fold — never on the full dataset.
    Original 3 from NB12 + 2 new v3 encodings.
    """
    global_mean  = train_df['PitNextLap'].mean()

    # ── Original 3 encodings (same as NB12) ────────────────────────────────────
    race_map    = train_df.groupby('Race')['PitNextLap'].mean()
    driver_map  = train_df.groupby('Driver')['PitNextLap'].mean()
    stint_len_map = (
        train_df.groupby(['Driver', 'Race', 'Year', 'Stint'])['TyreLife']
        .max().reset_index().groupby('Driver')['TyreLife'].mean()
    )
    global_stint = stint_len_map.mean()

    # ── New v3: Driver_stint_pct ───────────────────────────────────────────────
    # Median of max(TyreLife) per stint, grouped by (Driver, Compound)
    # Captures "what fraction of typical stint are we at?" — ratio form
    typical_stint = (
        train_df.groupby(['Driver', 'Race', 'Year', 'Stint', 'Compound'])['TyreLife']
        .max().reset_index()
        .groupby(['Driver', 'Compound'])['TyreLife']
        .median()
    )
    global_typical = typical_stint.mean()
    driver_cmpd_typical = typical_stint.to_dict()

    # ── New v3: Race_Compound_target_encoded ───────────────────────────────────
    race_cmpd_map = train_df.groupby(['Race', 'Compound'])['PitNextLap'].mean().to_dict()

    # ── Apply to val_df ────────────────────────────────────────────────────────
    out = val_df.copy()
    out['Race_target_encoded']     = out['Race'].map(race_map).fillna(global_mean)
    out['Driver_target_encoded']   = out['Driver'].map(driver_map).fillna(global_mean)
    out['Driver_avg_stint_length'] = out['Driver'].map(stint_len_map).fillna(global_stint)

    # Driver_stint_pct: TyreLife / typical_stint_for_Driver+Compound
    typical_vals = np.array([
        driver_cmpd_typical.get((d, c), global_typical)
        for d, c in zip(out['Driver'], out['Compound'])
    ], dtype=float)
    out['Driver_stint_pct'] = out['TyreLife'].values / np.maximum(typical_vals, 1.0)

    # Race_Compound_target_encoded
    out['Race_Compound_target_encoded'] = [
        race_cmpd_map.get((r, c), global_mean)
        for r, c in zip(out['Race'], out['Compound'])
    ]

    return out

print('apply_target_encodings_v3 defined.')

# Quick sanity check on fold 0
tr_idx  = np.where(train['fold'] != 0)[0]
val_idx = np.where(train['fold'] == 0)[0]
val_enc = apply_target_encodings_v3(train.iloc[tr_idx], train.iloc[val_idx])
print('Sample encodings (fold 0 val):')
new_enc_cols = ['Race_target_encoded', 'Driver_target_encoded', 'Driver_avg_stint_length',
                'Driver_stint_pct', 'Race_Compound_target_encoded']
print(val_enc[new_enc_cols].describe().round(4))

apply_target_encodings_v3 defined.
Sample encodings (fold 0 val):
       Race_target_encoded  Driver_target_encoded  Driver_avg_stint_length  \
count           88018.0000             88018.0000               88018.0000   
mean                0.1488                 0.1957                  20.8013   
std                 0.0918                 0.0565                   3.4491   
min                 0.0136                 0.0000                  11.8750   
25%                 0.0999                 0.1735                  19.1320   
50%                 0.1332                 0.1947                  20.7629   
75%                 0.1896                 0.2190                  21.8386   
max                 0.4074                 0.5783                  33.4643   

       Driver_stint_pct  Race_Compound_target_encoded  
count        88018.0000                    88018.0000  
mean             0.6816                        0.1487  
std              0.4843                        0.1474  
min    

In [5]:
# ── Feature sets ──────────────────────────────────────────────────────────────
BASE_FEATURES = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq',
    'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session',
    'Stint', 'Position',
    'Cumulative_Degradation_winsorized', 'Degradation_rate', 'Degradation_acceleration',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3',  'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
TARGET_ENC_ORIG = ['Race_target_encoded', 'Driver_target_encoded', 'Driver_avg_stint_length']
TARGET_ENC_NEW  = ['Driver_stint_pct', 'Race_Compound_target_encoded']
TIER14_FEATS = [
    'TyreLife_x_compound_ordinal', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem',
    'prime_pit_window', 'prime_window_x_compound',
    'abs_position_change', 'pos_change_rolling_std_3',
    'PitStop_lag1', 'laps_to_driver_end',
]
TIER7_NON_ENC = ['TyreLife_cliff_proximity', 'position_trend_slope', 'TyreLife_x_cliff_proximity']

# Baseline (38 features — same as NB12)
FEAT_COLS_38 = BASE_FEATURES + TARGET_ENC_ORIG + TIER14_FEATS
# V3 (43 features)
FEAT_COLS_43 = BASE_FEATURES + TARGET_ENC_ORIG + TARGET_ENC_NEW + TIER14_FEATS + TIER7_NON_ENC

assert len(FEAT_COLS_38) == 38, f'Expected 38, got {len(FEAT_COLS_38)}'
assert len(FEAT_COLS_43) == 43, f'Expected 43, got {len(FEAT_COLS_43)}'

# Verify all non-encoding features exist in parquets
non_enc_38 = BASE_FEATURES + TIER14_FEATS
non_enc_43 = non_enc_38 + TIER7_NON_ENC
missing_train = [f for f in non_enc_43 if f not in train.columns]
missing_test  = [f for f in non_enc_43 if f not in test.columns]
assert not missing_train, f'Missing in train: {missing_train}'
assert not missing_test,  f'Missing in test:  {missing_test}'

print(f'FEAT_COLS_38: {len(FEAT_COLS_38)} features (NB12 baseline)')
print(f'FEAT_COLS_43: {len(FEAT_COLS_43)} features (v3 — +5 new)')
print('All feature columns verified in parquets.')

FEAT_COLS_38: 38 features (NB12 baseline)
FEAT_COLS_43: 43 features (v3 — +5 new)
All feature columns verified in parquets.


In [6]:
# NB12 best LGBM params (baseline)
LGBM_NB12 = dict(
    n_estimators      = 3000,
    learning_rate     = 0.022005,
    num_leaves        = 31,
    min_child_samples = 12,
    subsample         = 0.967870,
    colsample_bytree  = 0.429823,
    reg_alpha         = 17.281674,
    reg_lambda        = 0.000018,
    max_bin           = 499,
    min_gain_to_split = 0.369129,
    random_state      = 42,
    verbose           = -1,
)

def run_cv(train_df, feature_cols, model_factory,
           test_df=None, lgbm_early_stopping=False, catboost_mode=False,
           target_enc_fn=None, label=''):
    """
    5-fold GroupKFold CV. Applies fold-aware target encodings inside each fold.
    Returns: (oof_auc, fold_aucs, oof_preds, test_preds_avg)
    """
    if target_enc_fn is None:
        target_enc_fn = apply_target_encodings_v3

    n   = len(train_df)
    oof = np.zeros(n)
    aucs = []
    test_fold_preds = []
    t0  = time.time()

    for fold_idx in range(5):
        tr_idx  = np.where(train_df['fold'] != fold_idx)[0]
        val_idx = np.where(train_df['fold'] == fold_idx)[0]

        train_enc = target_enc_fn(train_df.iloc[tr_idx], train_df.iloc[tr_idx])
        val_enc   = target_enc_fn(train_df.iloc[tr_idx], train_df.iloc[val_idx])

        missing = [f for f in feature_cols if f not in train_enc.columns]
        if missing:
            raise KeyError(f'Missing in fold {fold_idx}: {missing}')

        X_tr  = train_enc[feature_cols].to_numpy(dtype=np.float32)
        y_tr  = train_enc['PitNextLap'].to_numpy()
        X_val = val_enc[feature_cols].to_numpy(dtype=np.float32)
        y_val = val_enc['PitNextLap'].to_numpy()

        m = model_factory()
        if lgbm_early_stopping:
            callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
            m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)
        elif catboost_mode:
            m.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
        else:
            m.fit(X_tr, y_tr)

        oof[val_idx] = m.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, oof[val_idx])
        aucs.append(fold_auc)

        if test_df is not None:
            test_enc  = target_enc_fn(train_df.iloc[tr_idx], test_df)
            X_test    = test_enc[feature_cols].to_numpy(dtype=np.float32)
            test_fold_preds.append(m.predict_proba(X_test)[:, 1])

        extra = ''
        if lgbm_early_stopping or catboost_mode:
            extra = f'trees={m.best_iteration_}'
        print(f'  Fold {fold_idx}: AUC={fold_auc:.4f}  {extra}')

    oof_auc = roc_auc_score(train_df['PitNextLap'], oof)
    elapsed = time.time() - t0
    print(f'  ─── OOF AUC: {oof_auc:.4f}  std={np.std(aucs):.4f}  n_feat={len(feature_cols)}  ({elapsed:.0f}s)')
    if label:
        print(f'  ─── {label}')

    test_avg = np.mean(test_fold_preds, axis=0) if test_fold_preds else None
    return oof_auc, np.array(aucs), oof, test_avg

print('run_cv defined.')

run_cv defined.


## Step 1 — Baseline CV (38 features, NB12 params)

Reproduce NB12 OOF AUC = 0.9024. Uses `apply_target_encodings_v3` which includes the original 3 target encodings — the 2 new ones are excluded from FEAT_COLS_38. This is a direct comparison baseline.

In [7]:
print('Step 1 — Baseline CV (38 features, NB12 params)')
print(f'Expected OOF AUC: ~0.9024\n')

auc_baseline, aucs_baseline, oof_baseline, _ = run_cv(
    train, FEAT_COLS_38,
    model_factory=lambda: lgb.LGBMClassifier(**LGBM_NB12),
    lgbm_early_stopping=True,
    label='Baseline (NB12 params, 38 features)',
)

BASELINE_REF = 0.9024
print(f'  Δ vs NB12 reference ({BASELINE_REF}): {auc_baseline - BASELINE_REF:+.4f}')

Step 1 — Baseline CV (38 features, NB12 params)
Expected OOF AUC: ~0.9024

  Fold 0: AUC=0.9105  trees=1531
  Fold 1: AUC=0.8821  trees=219
  Fold 2: AUC=0.9058  trees=314
  Fold 3: AUC=0.9170  trees=1488
  Fold 4: AUC=0.8962  trees=1043
  ─── OOF AUC: 0.9024  std=0.0122  n_feat=38  (100s)
  ─── Baseline (NB12 params, 38 features)
  Δ vs NB12 reference (0.9024): -0.0000


## Step 2 — Tier 7 CV (43 features, NB12 params)

Add all 5 new features to measure their marginal contribution using the same NB12 params (no re-tuning yet). This gives a clean signal of the raw feature value.

In [8]:
print('Step 2 — Tier 7 CV (43 features, NB12 params)')
print(f'New features: {TIER7_NON_ENC + TARGET_ENC_NEW}\n')

auc_t7, aucs_t7, oof_t7, _ = run_cv(
    train, FEAT_COLS_43,
    model_factory=lambda: lgb.LGBMClassifier(**LGBM_NB12),
    lgbm_early_stopping=True,
    label='Tier 7 (NB12 params, 43 features)',
)

delta_t7 = auc_t7 - auc_baseline
print(f'\n  Δ vs 38-feature baseline: {delta_t7:+.4f}  ({"KEEP — re-tune" if delta_t7 >= 0 else "DROP — features hurt"})')
print(f'\nNote: reg_alpha=17.28 may still suppress some new features. Re-tuning with lower reg_alpha next.')

Step 2 — Tier 7 CV (43 features, NB12 params)
New features: ['TyreLife_cliff_proximity', 'position_trend_slope', 'TyreLife_x_cliff_proximity', 'Driver_stint_pct', 'Race_Compound_target_encoded']

  Fold 0: AUC=0.9070  trees=1542
  Fold 1: AUC=0.8594  trees=145
  Fold 2: AUC=0.8715  trees=235
  Fold 3: AUC=0.9066  trees=1596
  Fold 4: AUC=0.8882  trees=1563
  ─── OOF AUC: 0.8877  std=0.0189  n_feat=43  (112s)
  ─── Tier 7 (NB12 params, 43 features)

  Δ vs 38-feature baseline: -0.0147  (DROP — features hurt)

Note: reg_alpha=17.28 may still suppress some new features. Re-tuning with lower reg_alpha next.


## Step 3 — Pre-Season Testing Ablation

The 22,492 Pre-Season Testing rows (5% of train) have pit rate 14.7% vs ~20% for actual races. Hypothesis: excluding them may improve CV AUC on race folds. GroupKFold assigns these rows to specific folds — their removal may help or hurt depending on fold composition.

In [9]:
print('Step 3 — Pre-Season Testing Ablation')
train_no_testing = train[train['is_testing_session'] == 0].reset_index(drop=True)
n_testing = len(train) - len(train_no_testing)
print(f'Dropped {n_testing} testing rows  |  Train size: {len(train)} → {len(train_no_testing)}')

auc_no_test, aucs_no_test, _, _ = run_cv(
    train_no_testing, FEAT_COLS_43,
    model_factory=lambda: lgb.LGBMClassifier(**LGBM_NB12),
    lgbm_early_stopping=True,
    label='No Pre-Season Testing rows (43 features, NB12 params)',
)

delta_ablation = auc_no_test - auc_t7
print(f'\n  Δ vs full training set: {delta_ablation:+.4f}')
if delta_ablation >= 0.001:
    print('  → DROP testing rows from training — confirmed improvement')
    USE_TESTING_ROWS = False
else:
    print('  → KEEP testing rows — no reliable improvement from exclusion')
    USE_TESTING_ROWS = True
print(f'  USE_TESTING_ROWS = {USE_TESTING_ROWS}')

Step 3 — Pre-Season Testing Ablation
Dropped 22492 testing rows  |  Train size: 439140 → 416648
  Fold 0: AUC=0.9081  trees=1556
  Fold 1: AUC=0.8427  trees=134
  Fold 2: AUC=0.8656  trees=242
  Fold 3: AUC=0.9048  trees=1527
  Fold 4: AUC=0.8977  trees=1524
  ─── OOF AUC: 0.8857  std=0.0255  n_feat=43  (102s)
  ─── No Pre-Season Testing rows (43 features, NB12 params)

  Δ vs full training set: -0.0020
  → KEEP testing rows — no reliable improvement from exclusion
  USE_TESTING_ROWS = True


In [10]:
# Select training set for Optuna and final CV
train_optuna = train if USE_TESTING_ROWS else train_no_testing
print(f'Training set for Optuna: {len(train_optuna)} rows  (testing rows included: {USE_TESTING_ROWS})')

Training set for Optuna: 439140 rows  (testing rows included: True)


## Step 4 — Optuna Re-Tuning (43 features, lower reg_alpha ceiling)

Key changes from NB12 Optuna:
- **`reg_alpha` range: [0.5, 10.0]** (NB12 was [1e-8, 50.0], best was 17.28). Lower ceiling forces the model to use more features, allowing compound-interaction features (corr 0.26–0.30) to contribute.
- **`reg_lambda` range: [1e-4, 5.0]** (tighter — L2 not important at this scale)
- **43-feature set** (5 new features vs NB12's 38)
- **150 trials** on folds 0+3 (same multi-fold approach as NB12)
- **n_estimators = 3000** ceiling (NB12 used up to 1498 trees at lr=0.022; lower lr may need more)

Starting point: NB12 best params are seeded as trial 0 to warm-start the search.

In [11]:
# Pre-compute train/val splits for folds 0 and 3 (used inside every trial)
TUNE_FOLDS = [0, 3]
tune_data  = {}

for tf in TUNE_FOLDS:
    tr_idx  = np.where(train_optuna['fold'] != tf)[0]
    val_idx = np.where(train_optuna['fold'] == tf)[0]
    train_enc = apply_target_encodings_v3(train_optuna.iloc[tr_idx], train_optuna.iloc[tr_idx])
    val_enc   = apply_target_encodings_v3(train_optuna.iloc[tr_idx], train_optuna.iloc[val_idx])
    tune_data[tf] = (
        train_enc[FEAT_COLS_43].to_numpy(dtype=np.float32),
        train_enc['PitNextLap'].to_numpy(),
        val_enc[FEAT_COLS_43].to_numpy(dtype=np.float32),
        val_enc['PitNextLap'].to_numpy(),
    )

print(f'Tune data prepared for folds: {TUNE_FOLDS}')
for tf, (X_tr, y_tr, X_val, y_val) in tune_data.items():
    print(f'  Fold {tf}: train={X_tr.shape}  val={X_val.shape}')


def lgbm_objective_v3(trial):
    params = dict(
        n_estimators      = 3000,
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 20, 127),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 200),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.3, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 0.5, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
        max_bin           = trial.suggest_int('max_bin', 255, 511),
        min_gain_to_split = trial.suggest_float('min_gain_to_split', 0.0, 0.5),
        random_state      = 42,
        verbose           = -1,
    )
    aucs = []
    for tf in TUNE_FOLDS:
        X_tr, y_tr, X_val, y_val = tune_data[tf]
        m = lgb.LGBMClassifier(**params)
        callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)
        aucs.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return float(np.mean(aucs))


lgbm_study_v3 = optuna.create_study(
    direction = 'maximize',
    sampler   = optuna.samplers.TPESampler(seed=42),
)

# Seed with NB12 best params (warm-start)
seed_params = {
    'learning_rate': 0.022005, 'num_leaves': 31, 'min_child_samples': 12,
    'subsample': 0.967870, 'colsample_bytree': 0.429823,
    'reg_alpha': 9.0,  # clipped to new range ceiling
    'reg_lambda': 0.001, 'max_bin': 499, 'min_gain_to_split': 0.369129,
}
lgbm_study_v3.enqueue_trial(seed_params)

print('Starting LGBM Optuna v3 (150 trials, folds 0+3, reg_alpha ∈ [0.5, 10.0]) — ~90 min...')
t0 = time.time()
lgbm_study_v3.optimize(lgbm_objective_v3, n_trials=150, show_progress_bar=True)
print(f'\nBest avg AUC (folds 0+3): {lgbm_study_v3.best_value:.4f}  ({time.time()-t0:.0f}s)')
print('Best LGBM v3 params:')
for k, v in lgbm_study_v3.best_params.items():
    print(f'  {k}: {v:.6f}' if isinstance(v, float) else f'  {k}: {v}')

Tune data prepared for folds: [0, 3]
  Fold 0: train=(351122, 43)  val=(88018, 43)
  Fold 3: train=(351286, 43)  val=(87854, 43)
Starting LGBM Optuna v3 (150 trials, folds 0+3, reg_alpha ∈ [0.5, 10.0]) — ~90 min...


Best trial: 99. Best value: 0.907823: 100%|██████████| 150/150 [1:04:39<00:00, 25.87s/it]


Best avg AUC (folds 0+3): 0.9078  (3880s)
Best LGBM v3 params:
  learning_rate: 0.048469
  num_leaves: 25
  min_child_samples: 175
  subsample: 0.679967
  colsample_bytree: 0.546971
  reg_alpha: 8.897172
  reg_lambda: 0.000129
  max_bin: 406
  min_gain_to_split: 0.142256


## Step 5 — Full 5-Fold CV with Re-Tuned Params

In [12]:
LGBM_V3_TUNED = dict(
    n_estimators = 3000,
    random_state = 42,
    verbose      = -1,
    **lgbm_study_v3.best_params,
)

print('LGBM v3 re-tuned — full 5-fold CV')
print(f'Params: {LGBM_V3_TUNED}\n')

auc_lgbm_v3, aucs_lgbm_v3, oof_lgbm_v3, test_lgbm_v3 = run_cv(
    train_optuna, FEAT_COLS_43,
    model_factory=lambda: lgb.LGBMClassifier(**LGBM_V3_TUNED),
    test_df=test,
    lgbm_early_stopping=True,
    label='LGBM v3 re-tuned (NB14 Optuna, 43 features)',
)

print(f'\n  Δ vs NB12 baseline (0.9024):  {auc_lgbm_v3 - 0.9024:+.4f}')
print(f'  Δ vs NB13 rank avg (0.9032):  {auc_lgbm_v3 - 0.9032:+.4f}')
print(f'  Projected LB (CV + 0.025):   ~{auc_lgbm_v3 + 0.025:.4f}')

LGBM v3 re-tuned — full 5-fold CV
Params: {'n_estimators': 3000, 'random_state': 42, 'verbose': -1, 'learning_rate': 0.04846947023716295, 'num_leaves': 25, 'min_child_samples': 175, 'subsample': 0.6799673654066314, 'colsample_bytree': 0.546970580714255, 'reg_alpha': 8.897171658612159, 'reg_lambda': 0.00012939148744696297, 'max_bin': 406, 'min_gain_to_split': 0.14225574185778586}

  Fold 0: AUC=0.9080  trees=709
  Fold 1: AUC=0.8565  trees=97
  Fold 2: AUC=0.8736  trees=97
  Fold 3: AUC=0.9077  trees=1113
  Fold 4: AUC=0.8882  trees=1220
  ─── OOF AUC: 0.8882  std=0.0199  n_feat=43  (89s)
  ─── LGBM v3 re-tuned (NB14 Optuna, 43 features)

  Δ vs NB12 baseline (0.9024):  -0.0142
  Δ vs NB13 rank avg (0.9032):  -0.0150
  Projected LB (CV + 0.025):   ~0.9132


## Step 6 — CatBoost NB11 Params on 43-Feature Set (Optional Quick Check)

CatBoost NB11 (depth=8, lr=0.05, default params) achieved 0.9016 on 38 features. Check if the 5 new features also help CatBoost with the same params. If yes, it strengthens the case for NB15's ensemble.

In [13]:
if not CATBOOST_AVAILABLE:
    print('CatBoost not available — skipping.')
    auc_cb_v3 = None
    oof_cb_v3 = None
    test_cb_v3 = None
else:
    # NB11 CatBoost-Plain params (unchanged — just checking new features)
    CB_NB11_PARAMS = dict(
        iterations     = 2000,
        depth          = 8,
        learning_rate  = 0.05,
        l2_leaf_reg    = 3.0,
        boosting_type  = 'Plain',
        eval_metric    = 'AUC',
        od_type        = 'Iter',
        od_wait        = 50,
        use_best_model = True,
        verbose        = 0,
        random_seed    = 42,
    )

    print('CatBoost NB11 params on 43-feature set (quick check)\n')
    auc_cb_v3, aucs_cb_v3, oof_cb_v3, test_cb_v3 = run_cv(
        train_optuna, FEAT_COLS_43,
        model_factory=lambda: CatBoostClassifier(**CB_NB11_PARAMS),
        test_df=test,
        catboost_mode=True,
        label='CatBoost NB11 params on 43 features',
    )

    CB_BASELINE_REF = 0.9016
    print(f'\n  Δ vs NB11 CB baseline (0.9016): {auc_cb_v3 - CB_BASELINE_REF:+.4f}')
    if oof_cb_v3 is not None:
        rho, _ = spearmanr(oof_lgbm_v3, oof_cb_v3)
        print(f'  Spearman ρ (LGBM v3 × CB v3): {rho:.4f}')

CatBoost NB11 params on 43-feature set (quick check)

  Fold 0: AUC=0.9052  trees=262
  Fold 1: AUC=0.8808  trees=3
  Fold 2: AUC=0.8868  trees=2
  Fold 3: AUC=0.9053  trees=742
  Fold 4: AUC=0.8878  trees=617
  ─── OOF AUC: 0.8524  std=0.0102  n_feat=43  (128s)
  ─── CatBoost NB11 params on 43 features

  Δ vs NB11 CB baseline (0.9016): -0.0492
  Spearman ρ (LGBM v3 × CB v3): 0.8366


## Step 7 — Results Summary

In [14]:
print('=' * 75)
print('NOTEBOOK 14 — RESULTS SUMMARY')
print('=' * 75)
print(f'  {"Step":<45} {"OOF AUC":>9}  {"Δ Baseline":>11}')
print('-' * 75)

rows = [
    ('NB12 baseline (38 feat, NB12 params)',        auc_baseline,  0.0),
    ('Tier 7 (43 feat, NB12 params)',               auc_t7,        auc_t7 - auc_baseline),
    ('No-testing ablation (43 feat)',               auc_no_test,   auc_no_test - auc_t7),
    ('LGBM v3 re-tuned (43 feat)',                  auc_lgbm_v3,   auc_lgbm_v3 - auc_baseline),
]
if auc_cb_v3 is not None:
    rows.append(('CatBoost NB11 params (43 feat)',  auc_cb_v3,     auc_cb_v3 - CB_BASELINE_REF))

for name, auc, delta in rows:
    delta_str = f'{delta:+.4f}' if delta != 0.0 else '  ref'
    print(f'  {name:<45} {auc:>9.4f}  {delta_str:>11}')

print('=' * 75)
print(f'  Best NB14 CV AUC:  {auc_lgbm_v3:.4f}')
print(f'  Projected LB AUC:  ~{auc_lgbm_v3 + 0.025:.4f}  (using +0.025 boost)')
print(f'  Gap to leaderboard top (0.95488): {0.95488 - (auc_lgbm_v3 + 0.025):+.4f} LB  |  {0.9032 - auc_lgbm_v3 + 0.027:+.4f} CV needed')

NOTEBOOK 14 — RESULTS SUMMARY
  Step                                            OOF AUC   Δ Baseline
---------------------------------------------------------------------------
  NB12 baseline (38 feat, NB12 params)             0.9024          ref
  Tier 7 (43 feat, NB12 params)                    0.8877      -0.0147
  No-testing ablation (43 feat)                    0.8857      -0.0020
  LGBM v3 re-tuned (43 feat)                       0.8882      -0.0142
  CatBoost NB11 params (43 feat)                   0.8524      -0.0492
  Best NB14 CV AUC:  0.8882
  Projected LB AUC:  ~0.9132  (using +0.025 boost)
  Gap to leaderboard top (0.95488): +0.0417 LB  |  +0.0420 CV needed


## Step 8 — Save Outputs

In [ ]:
# ── Save v3 parquets (base columns + Tier 7 non-encoding features) ─────────
TIER7_COLS = ['TyreLife_cliff_proximity', 'position_trend_slope', 'TyreLife_x_cliff_proximity']

# Preserve all v2 columns + add Tier 7 non-enc features
v2_train_cols = [c for c in pd.read_parquet(processed_dir / 'train_features_v2.parquet').columns]
KEEP_TRAIN_V3 = v2_train_cols + [c for c in TIER7_COLS if c not in v2_train_cols]
KEEP_TEST_V3  = [c for c in KEEP_TRAIN_V3 if c != 'PitNextLap']

# Reconstruct full train/test with v3 features (train_optuna may have dropped testing rows)
# Save full train (not the ablated version) for downstream notebooks
train_save = train[KEEP_TRAIN_V3]
test_save  = test[KEEP_TEST_V3]

v3_train_path = processed_dir / 'train_features_v3.parquet'
v3_test_path  = processed_dir / 'test_features_v3.parquet'
train_save.to_parquet(v3_train_path, index=False)
test_save.to_parquet(v3_test_path,   index=False)
print(f'Saved train_features_v3: {v3_train_path}  shape={train_save.shape}')
print(f'Saved test_features_v3 : {v3_test_path}   shape={test_save.shape}')

# ── Save OOF predictions ──────────────────────────────────────────────────
# Use train_optuna index — map back to id for alignment
oof_df = pd.DataFrame({
    'id':         train_optuna['id'].values,
    'fold':       train_optuna['fold'].values,
    'PitNextLap': train_optuna['PitNextLap'].values,
    'lgbm_v3_oof': oof_lgbm_v3,
})
if oof_cb_v3 is not None:
    oof_df['cb_v3_oof'] = oof_cb_v3

oof_path = processed_dir / 'oof_predictions_nb14.parquet'
oof_df.to_parquet(oof_path, index=False)
print(f'Saved OOF: {oof_path}  shape={oof_df.shape}')

# ── Save test predictions ────────────────────────────────────────────────
test_df_out = pd.DataFrame({'id': test['id'].values})
if test_lgbm_v3 is not None:
    test_df_out['lgbm_v3_pred'] = test_lgbm_v3
if test_cb_v3 is not None:
    test_df_out['cb_v3_pred'] = test_cb_v3

test_path = processed_dir / 'test_predictions_nb14.parquet'
test_df_out.to_parquet(test_path, index=False)
print(f'Saved test: {test_path}  shape={test_df_out.shape}')

# ── Save summary pkl for NB16/17 ─────────────────────────────────────────
summary = {
    'lgbm_v3_tuned_params': LGBM_V3_TUNED,
    'lgbm_v3_auc':          auc_lgbm_v3,
    'lgbm_v3_fold_aucs':    aucs_lgbm_v3,
    'auc_baseline_38feat':  auc_baseline,
    'auc_tier7_43feat':     auc_t7,
    'auc_no_testing':       auc_no_test,
    'use_testing_rows':     USE_TESTING_ROWS,
    'feature_cols_43':      FEAT_COLS_43,
    'feature_cols_38':      FEAT_COLS_38,
    'tier7_non_enc':        TIER7_NON_ENC,
    'tier7_enc_new':        TARGET_ENC_NEW,
    'optuna_best_value':    lgbm_study_v3.best_value,
}
if auc_cb_v3 is not None:
    summary.update({
        'cb_v3_auc':       auc_cb_v3,
        'cb_v3_fold_aucs': aucs_cb_v3,
        'rho_lgbm_cb_v3':  float(spearmanr(oof_lgbm_v3, oof_cb_v3)[0]) if oof_cb_v3 is not None else None,
    })

summary_path = models_dir / 'nb14_summary.pkl'
with open(summary_path, 'wb') as f:
    pickle.dump(summary, f)
print(f'Saved summary: {summary_path}')
print(f'Keys: {list(summary.keys())}')

## Summary

### What NB14 Does

| Step | Action | Decision criterion |
|------|--------|-------------------|
| 1 | Baseline CV (38 feat, NB12 params) | Reproduce 0.9024 |
| 2 | Tier 7 CV (43 feat, NB12 params) | Δ ≥ 0 → keep new features |
| 3 | Pre-Season Testing ablation | Δ ≥ +0.001 → drop testing rows |
| 4 | Optuna re-tune (reg_alpha ∈ [0.5, 10.0]) | 150 trials, folds 0+3 |
| 5 | Full 5-fold CV with tuned params | Report OOF AUC |
| 6 | CatBoost quick check (NB11 params) | Δ vs 0.9016 |

### Next Step

**Notebook 15 — Neural Network (MLP with Entity Embeddings).** Goal: achieve Spearman ρ < 0.90 vs LGBM and solo AUC ≥ 0.895 to break the GBM correlation ceiling.